### Implement Your Own Retriever

In [1]:
docs = [
    "RAG stands for Retrieval-Augmented Generation.",
    "It combines information retrieval and text generation.",
    "LangChain can be used to build RAG pipelines easily.",
    "Transformers use self-attention to handle sequential data."
]

query = "What is RAG?"

#TODO COMPLETED: Keyword-overlap retriever
def retrieve_context(query, docs, k=1):
    """
    Keyword-based retriever.

    How it works:
      1. Split the query into individual lowercase words (a set, so duplicates are ignored)
      2. For each document, count how many query words appear inside it
         → this count is the 'relevance score'
      3. Sort all documents by score (highest first)
      4. Return the top-k most relevant documents

    Example:
      query = "What is RAG?"
      query_words = {'what', 'is', 'rag?'}          ← query split into words
      doc = "RAG stands for Retrieval-Augmented..."  ← contains 'rag' → score 1
      doc = "Transformers use self-attention..."     ← no match      → score 0
    """
    # Step 1: Break the query into a set of lowercase words
    query_words = set(query.lower().split())

    # Step 2: Score every document
    scored_docs = []
    for doc in docs:
        doc_words = set(doc.lower().split())          # doc words as a set
        score = len(query_words & doc_words)          # & = intersection = matching words
        scored_docs.append((score, doc))

    # Step 3: Sort by score descending → highest relevance first
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    # Step 4: Return only the text of the top-k documents
    return [doc for _, doc in scored_docs[:k]]


context = retrieve_context(query, docs)
print("Retrieved context:", context)
# Expected output: ['RAG stands for Retrieval-Augmented Generation.']
# Reason: this doc contains both 'rag' and 'is', scoring 2 matching words.

Retrieved context: ['RAG stands for Retrieval-Augmented Generation.']


### Write the Prompt Template

In [2]:
#TODO COMPLETED: Prompt template builder
def make_prompt(context, question):
    """
    Assembles a structured prompt for the LLM.

    Why this structure matters:
      - 'Context:' tells the model where to look for the answer.
      - 'Question:' focuses the model on what to answer.
      - 'Answer:' acts as a trigger — the model continues from this token,
        generating the answer text. Without it, the model would continue
        the question rather than switch to answering mode.

    Args:
        context  : list[str] or str — retrieved document(s)
        question : str              — the user's query

    Returns:
        str — the complete prompt string ready for the LLM
    """
    # context may be a list (retrieve_context returns a list) → join into one string
    if isinstance(context, list):
        context_str = "\n".join(context)
    else:
        context_str = context

    return f"Context: {context_str}\nQuestion: {question}\nAnswer:"


prompt_text = make_prompt(context, query)
print(prompt_text)
# Expected output:
# Context: RAG stands for Retrieval-Augmented Generation.
# Question: What is RAG?
# Answer:

Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer:


## Plug in a HuggingFace Model
Experiment with temperature (0.2, 0.8) and observe changes.

In [3]:
from transformers import pipeline

# TODO COMPLETED: Initialize the text-generation pipeline
# pad_token_id=50256 suppresses the padding warning (uses <|endoftext|> as pad token)
hf_gen = pipeline(
    task="text-generation",
    model="distilgpt2",       # ← was ???
    max_new_tokens=50,        # ← was ???
    pad_token_id=50256,
)

#TODO COMPLETED: Generate an answer using the prompt built above
response = hf_gen(prompt_text)[0]["generated_text"]
print(response)

c:\Users\mohit\Documents\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1831.13it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing

Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer: RAG is a short term prediction algorithm for real-time computing. It is used by companies such as IBM, Intel, Intel, Intel, Microsoft, IBM, Intel, Qualcomm, Qualcomm, Qualcomm, Qualcomm and so on. For example, a


### Temperature Experiment: 0.2 vs 0.8

**What temperature controls:**  
Temperature scales the probability distribution over the model's vocabulary before sampling the next token.

| Temperature | Effect | When to use |
|---|---|---|
| **Low (0.2)** | Distribution is sharp → model picks the highest-probability token almost every time → output is focused, repetitive, deterministic | Factual Q&A, RAG, code generation |
| **High (0.8)** | Distribution is flat → model samples more randomly → output is creative, varied, sometimes incoherent | Brainstorming, creative writing |

In [4]:
# Temperature experiment — run the same prompt with two different temperatures

# do_sample=True is required whenever temperature != 1.0
# Without it, HuggingFace ignores the temperature and uses greedy decoding.

gen_low = pipeline(
    "text-generation", model="distilgpt2",
    max_new_tokens=50, temperature=0.2,
    do_sample=True, pad_token_id=50256
)

gen_high = pipeline(
    "text-generation", model="distilgpt2",
    max_new_tokens=50, temperature=0.8,
    do_sample=True, pad_token_id=50256
)

test_prompt = make_prompt(context, query)

print("=" * 55)
print("TEMPERATURE = 0.2  (focused / deterministic)")
print("=" * 55)
print(gen_low(test_prompt)[0]["generated_text"])

print()
print("=" * 55)
print("TEMPERATURE = 0.8  (creative / random)")
print("=" * 55)
print(gen_high(test_prompt)[0]["generated_text"])

# Observation:
# temperature=0.2 → same or very similar output every run
# temperature=0.8 → output changes noticeably between runs

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1342.25it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 2008.79it/s]


TEMPERATURE = 0.2  (focused / deterministic)


[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer: RAG is a new type of RAG that is used to make the data available to the user. RAG is a new type of RAG that is used to make the data available to the user. RAG is a new type of R

TEMPERATURE = 0.8  (creative / random)
Context: RAG stands for Retrieval-Augmented Generation.
Question: What is RAG?
Answer: RAG is a new feature that is being released in 2017. It allows us to create more complex and more complex objects, as well as more generic objects and more precise expressions. The RAG concept is based on the principles of the RAG concept


## Connect It All: Build a Mini RAG Function

In [5]:
# TODO COMPLETED: mini_rag wires together all three steps:
#   retrieve_context  → finds the most relevant document(s)
#   make_prompt       → builds the structured prompt string
#   hf_gen            → generates the answer

def mini_rag(query):
    """
    Full RAG pipeline in 3 lines:
      1. RETRIEVE  — find the most relevant doc(s) for the query
      2. AUGMENT   — inject them as context into a structured prompt
      3. GENERATE  — let the LLM produce an answer grounded in that context
    """
    context = retrieve_context(query, docs)   # Step 1 — Retrieve
    prompt  = make_prompt(context, query)     # Step 2 — Augment
    result  = hf_gen(prompt)[0]["generated_text"]  # Step 3 — Generate
    return result


# ── Test with two different queries ───────────────────────────────
print("=" * 55)
print("QUERY: 'How do transformers work?'")
print("=" * 55)
print(mini_rag("How do transformers work?"))
# The retriever will score 'Transformers use self-attention...' highest
# because 'transformers' appears in both the query and that document.

print()
print("=" * 55)
print("QUERY: 'What is LangChain used for?'")
print("=" * 55)
print(mini_rag("What is LangChain used for?"))
# The retriever will surface 'LangChain can be used to build RAG pipelines easily.'

QUERY: 'How do transformers work?'


[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: Transformers use self-attention to handle sequential data.
Question: How do transformers work?
Answer: If you do the same thing in your library, you are not going to be able to do it easily if you are using a different type of data.
Question: Is the system really the same?
Answer: We don't really have to

QUERY: 'What is LangChain used for?'
Context: LangChain can be used to build RAG pipelines easily.
Question: What is LangChain used for?
Answer: LangChain is the default solution for RAG pipeline and is not used for RAG pipelines.
Question: What are the RAG pipelines?
Answer: LangChain is the one that needs to be built in.
Question: What is Lang
